In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, count, when, isnan,log1p

In [0]:
df = spark.read.table('especializacion.gold_zone.data_kagle')
display(df)

In [0]:
# Total de registros
print(f"Total registros: {df.count()}")

# Cuántos tienen rating y cuántos no
df.select(
    count(when(col("calificacion_producto").isNull(), 1)).alias("sin_rating"),
    count(when(col("calificacion_producto").isNotNull(), 1)).alias("con_rating")
).display()

# Distribución del rating
df.groupBy("calificacion_producto").count().orderBy("calificacion_producto").display()

In [0]:


# Seleccionar columnas relevantes
# Ajusta los nombres según el CSV real
df = df.select(
    "nombre_producto",
    "categoria_principal",
    "sub_categoria",
    "precio",
    "cantidad_calificaciones",
    "pais",
    "currency",
    "calificacion_producto"
)

# # Castear tipos si es necesario
# df = df.withColumn("price", col("price").cast("double")) \
#        .withColumn("num_reviews", col("num_reviews").cast("double")) \
#        .withColumn("rating", col("rating").cast("double"))

# Feature: log de reviews (reduce sesgo por outliers)
df = df.withColumn("log_reviews", log1p(col("cantidad_calificaciones")))

# Imputar precio nulo con la mediana (simple)
from pyspark.sql.functions import percentile_approx
median_price = df.select(
    percentile_approx("precio", 0.5).alias("med")
).collect()[0]["med"]

df = df.fillna({"precio": median_price, "cantidad_calificaciones": 0, "log_reviews": 0})

df.display()

In [0]:
# Con rating para entrenar y evaluar
df_labeled   = df.filter(col("calificacion_producto").isNotNull())

# Sin rating para predecir
df_unlabeled = df.filter(col("calificacion_producto").isNull())

print(f"Con rating (train/test): {df_labeled.count()}")
print(f"Sin rating (a predecir): {df_unlabeled.count()}")

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# ── 1. Indexar columnas categóricas ──────────────────────────────────────
cat_cols = ["categoria_principal", "sub_categoria", "pais"]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_cols
]

encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in cat_cols
]

# ── 2. Ensamblar vector de features ──────────────────────────────────────
num_cols     = ["precio", "log_reviews"]
encoded_cols = [f"{c}_ohe" for c in cat_cols]

assembler = VectorAssembler(
    inputCols=num_cols + encoded_cols,
    outputCol="features_raw",
    handleInvalid="skip"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=False, withStd=True
)

# ── 3. Modelo ─────────────────────────────────────────────────────────────
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="calificacion_producto",
    numTrees=100,
    maxDepth=6,
    seed=42
)

# ── 4. Pipeline ───────────────────────────────────────────────────────────
pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, rf])

In [0]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# Split 80/20 sobre los datos con rating
train_df, test_df = df_labeled.randomSplit([0.8, 0.2], seed=42)

print(f"Train: {train_df.count()} | Test: {test_df.count()}")

# Entrenar
model = pipeline.fit(train_df)
print("✅ Modelo entrenado correctamente")

In [0]:
evaluator_rmse = RegressionEvaluator(
    labelCol="calificacion_producto", predictionCol="prediction", metricName="rmse"
)
evaluator_r2 = RegressionEvaluator(
    labelCol="calificacion_producto", predictionCol="prediction", metricName="r2"
)
evaluator_mae = RegressionEvaluator(
    labelCol="calificacion_producto", predictionCol="prediction", metricName="mae"
)

predictions = model.transform(test_df)

rmse = evaluator_rmse.evaluate(predictions)
r2   = evaluator_r2.evaluate(predictions)
mae  = evaluator_mae.evaluate(predictions)

print(f"📊 RMSE : {rmse:.4f}")
print(f"📊 R²   : {r2:.4f}")
print(f"📊 MAE  : {mae:.4f}")

# Comparar predicción vs real
predictions.select("nombre_producto", "calificacion_producto", "prediction") \
            .orderBy("calificacion_producto") \
            .display()

In [0]:
# Aplicar el modelo sobre los productos sin rating
df_predicted = model.transform(df_unlabeled)

# Resultado final: clamp entre 1.0 y 5.0 (rango válido de ratings)
from pyspark.sql.functions import least, greatest, lit, round as spark_round

df_final = df_predicted.withColumn(
    "rating_predicho",
    spark_round(
        least(lit(5.0), greatest(lit(1.0), col("prediction"))),
        2
    )
)

df_final.select(
    "nombre_producto", "categoria_principal", "precio", "pais", "rating_predicho"
).display()

In [0]:
df.columns

In [0]:
# Guardar predicciones en una tabla Delta
df_final.select(
    "nombre_producto", "categoria_principal", "sub_categoria",
    "precio", "pais", "rating_predicho"
).write \
 .format("delta") \
 .mode("overwrite") \
 .saveAsTable("especializacion.gold_zone.ikea_ratings_predichos")

print("Resultados guardados en tabla Delta: ikea_ratings_predichos")